In [ ]:
#####################################################
#
# APLICAR Regresión lineal a datos preprocesados con PCA
# y elegir el mejor modelo después de regularizacion
#
#####################################################
# Deben cargarse los archivos
# - T_train_final_objetivo.csv
# - T_test_final_objetivo.csv"
#
# Devolverá
# - reg_lin_ganador_bundle.zip
#####################################################

import pandas as pd
import numpy as np
import math
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# === Carga de datos ===
Train = pd.read_csv("T_train_final_objetivo.csv")
Test = pd.read_csv("T_test_final_objetivo.csv")

X_test = Test.iloc[:, :-1]
X_train = Train.iloc[:, :-1]
y_train = Train.iloc[:, -1].to_numpy(dtype=float)
y_test = Test.iloc[:, -1].to_numpy(dtype=float)

SEP = "___"

In [ ]:
#############################
# Funciones auxiliares
#############################

def is_binary_series(s: pd.Series):
    vals = pd.unique(s.dropna())
    return set(vals).issubset({0,1}) or set(vals).issubset({0.0,1.0})

def prefix_of(col: str, sep=SEP):
    return col.split(sep, 1)[0] if sep in col else None

def build_nominal_blocks_by_prefix(X: pd.DataFrame, sep=SEP):
    blocks = {}
    for c in X.columns:
        if sep in c and is_binary_series(X[c]):
            blocks.setdefault(prefix_of(c, sep), []).append(c)
    for k,v in blocks.items():
        blocks[k] = [c for c in X.columns if c in set(v)]
    return blocks


In [ ]:
#############################
# Construcción del pipeline base
#############################

blocks = build_nominal_blocks_by_prefix(X_train, SEP)
drop_cols = [cols[0] for cols in blocks.values() if len(cols) >= 2]

arreglar_despeje = ColumnTransformer(
    transformers=[("drop_nominal_bases", "drop", drop_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False,
    force_int_remainder_cols=False
)

mi_regresion_lineal = Pipeline([
    ("dropper", arreglar_despeje),
    ("linreg", LinearRegression(fit_intercept=True)),
])

mi_regresion_lineal.fit(X_train, y_train)

In [ ]:
#############################
# Coeficientes e intercepto
#############################
intercepto = mi_regresion_lineal.named_steps["linreg"].intercept_
coefs = mi_regresion_lineal.named_steps["linreg"].coef_
feature_names = mi_regresion_lineal.named_steps["dropper"].get_feature_names_out(X_train.columns)

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
print("Intercepto (beta0):", intercepto)
print(coef_df)

In [ ]:
#############################
# Evaluación básica en Train y Test
#############################
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

yhat_test = mi_regresion_lineal.predict(X_test)
r2_test   = r2_score(y_test, yhat_test)
rmse_test = math.sqrt(mean_squared_error(y_test, yhat_test))
mae_test  = mean_absolute_error(y_test, yhat_test)

yhat_train = mi_regresion_lineal.predict(X_train)
r2_train   = r2_score(y_train, yhat_train)
rmse_train = math.sqrt(mean_squared_error(y_train, yhat_train))
mae_train  = mean_absolute_error(y_train, yhat_train)

print({"R2_test": r2_test, "RMSE_test": rmse_test, "MAE_test": mae_test})
print({"R2_train": r2_train, "RMSE_train": rmse_train, "MAE_train": mae_train})


In [ ]:
# Veredicto

# NRMSE y mejora vs media en TEST
std_y_test = float(np.std(y_test, ddof=0))
nrmse_test = rmse_test / (std_y_test + 1e-12)
mejora_pct = 100.0 * (1.0 - nrmse_test)  # % mejor que predecir la media

# Semáforo "aquí dentro"
verde   = (r2_test >= 0.70) and (nrmse_test <= 0.50)
amarilo = (0.40 <= r2_test < 0.70) or (0.50 < nrmse_test <= 0.80)
if verde:
    veredicto = "VERDE"
    significado = "confiable para predecir aquí dentro."
elif amarilo:
    veredicto = "AMARILLO"
    significado = "usable con cautela (depende del caso de uso)"
else:
    veredicto = "ROJO"
    significado = "no confiable para predicción aquí dentro"

# Generar explicación humanizada automáticamente
def generar_explicacion(r2_test, nrmse_test, mejora_pct, rmse_test, mae_test, veredicto):
    explicacion = []

    # Explicación basada en R²
    if r2_test >= 0.8:
        explicacion.append(f"• El modelo explica un {r2_test:.1%} de la variabilidad en los datos, lo que indica un excelente ajuste.")
        explicacion.append("  🎯 **Analogía**: Como un pronóstico del tiempo que acierta 8 de cada 10 días - muy confiable.")
    elif r2_test >= 0.6:
        explicacion.append(f"• El modelo captura un {r2_test:.1%} de la variabilidad, mostrando una buena capacidad predictiva.")
        explicacion.append("  🎯 **Analogía**: Similar a un forecast económico que identifica correctamente las tendencias principales.")
    elif r2_test >= 0.4:
        explicacion.append(f"• Con un R² del {r2_test:.1%}, el modelo tiene capacidad predictiva moderada.")
        explicacion.append("  🎯 **Analogía**: Como un detector de lluvia que funciona bien para saber si lloverá, pero no cuánto.")
    else:
        explicacion.append(f"• El R² de {r2_test:.1%} sugiere que el modelo tiene capacidad predictiva limitada.")
        explicacion.append("  🎯 **Analogía**: Parecido a adivinar el clima lanzando una moneda - mejor que nada, pero poco confiable.")

    # Explicación basada en NRMSE
    if nrmse_test <= 0.3:
        explicacion.append(f"• Los errores de predicción son muy bajos ({nrmse_test:.1%} de la variabilidad total).")
        explicacion.append("  📏 **Analogía**: Como medir con una regla milimetrada - alta precisión en las estimaciones.")
    elif nrmse_test <= 0.5:
        explicacion.append(f"• Los errores son moderados ({nrmse_test:.1%} de la variabilidad total).")
        explicacion.append("  📏 **Analogía**: Similar a usar una cinta métrica - útil para la mayoría de propósitos prácticos.")
    elif nrmse_test <= 0.7:
        explicacion.append(f"• Los errores son considerables ({nrmse_test:.1%} de la variabilidad total).")
        explicacion.append("  📏 **Analogía**: Como estimar distancias a ojo - sirve para aproximaciones gruesas.")
    else:
        explicacion.append(f"• Los errores son muy altos ({nrmse_test:.1%} de la variabilidad total).")
        explicacion.append("  📏 **Analogía**: Parecido a adivinar el tamaño de algo desde lejos - muy impreciso.")

    # Explicación basada en mejora vs media
    if mejora_pct > 50:
        explicacion.append(f"• Es un {mejora_pct:.0f}% mejor que simplemente predecir el promedio, una mejora sustancial.")
        explicacion.append("  🚀 **Analogía**: Como usar GPS vs. solo un mapa de carreteras - mucho más eficiente.")
    elif mejora_pct > 20:
        explicacion.append(f"• Mejora en un {mejora_pct:.0f}% respecto a predecir la media.")
        explicacion.append("  🚀 **Analogía**: Similar a tener indicaciones de tráfico en tiempo real - claramente mejor que sin ellas.")
    elif mejora_pct > 0:
        explicacion.append(f"• Solo un {mejora_pct:.0f}% mejor que predecir el promedio, mejora marginal.")
        explicacion.append("  🚀 **Analogía**: Como tener una brújula en lugar de solo el norte - ayuda, pero no demasiado.")
    else:
        explicacion.append("• No mejora respecto a predecir el valor promedio.")
        explicacion.append("  🚀 **Analogía**: Como intentar navegar sin brújula ni mapa - no aporta ventaja.")

    # Comparación entre train y test (si estuvieran disponibles ambos)
    if 'r2_train' in locals():
        sobreajuste = r2_train - r2_test
        if sobreajuste > 0.2:
            explicacion.append(f"• Hay indicios de sobreajuste (R² train: {r2_train:.3f} vs test: {r2_test:.3f}).")
            explicacion.append("  ⚠️ **Analogía**: Como un estudiante que memoriza las respuestas pero no entiende el concepto.")
        elif sobreajuste < 0.05:
            explicacion.append("• El modelo generaliza bien, sin signos evidentes de sobreajuste.")
            explicacion.append("  ✅ **Analogía**: Similar a un atleta que entrena y compite igual de bien - consistente.")

    # Interpretación del veredicto con analogías finales
    if veredicto == "VERDE":
        explicacion.append("\n✅ **Conclusión**: El modelo es confiable para hacer predicciones en contextos similares a los datos de prueba.")
        explicacion.append("🎯 **Analogía final**: Como un piloto automático confiable - puedes usarlo para navegar con seguridad.")
    elif veredicto == "AMARILLO":
        explicacion.append("\n⚠️ **Conclusión**: Úsalo con precaución - puede ser útil para identificar tendencias pero no para predicciones precisas.")
        explicacion.append("🎯 **Analogía final**: Como el forecast de fin de semana - útil para planear, pero lleva paraguas por si acaso.")
    else:
        explicacion.append("\n❌ **Conclusión**: Se recomienda revisar las variables o considerar modelos alternativos.")
        explicacion.append("🎯 **Analogía final**: Como un mapa muy antiguo - mejor conseguir uno actualizado o usar otros métodos.")

    return "\n".join(explicacion)

# Generar la explicación
explicacion_humanizada = generar_explicacion(
    r2_test, nrmse_test, mejora_pct, rmse_test, mae_test, veredicto
)

resumen = {
    "R2_test": r2_test,
    "MAE_test": mae_test,
    "RMSE_test": rmse_test,
    "std(y_test)": std_y_test,
    "NRMSE_test": nrmse_test,
    "Mejora_vs_media_%": mejora_pct,
    "Veredicto": veredicto,
    "significado": significado
}

df_resumen = pd.DataFrame(resumen, index=[0]).T
df_resumen.columns = ["Valor"]
df_resumen.index.name = "Métrica"

print(df_resumen)
print("\n" + "="*60)
print("EXPLICACIÓN DEL VEREDICTO:")
print("="*60)
print(explicacion_humanizada)

def explicacion_breve(r2_test, nrmse_test, mejora_pct, veredicto):
    base = f"Con un R² de {r2_test:.3f} y un error relativo (NRMSE) de {nrmse_test:.3f}, "

    if veredicto == "VERDE":
        analogia = "Como un GPS confiable - puedes seguir sus indicaciones con seguridad."
        return base + f"el modelo es robusto y explica bien los patrones en los datos, siendo {mejora_pct:.0f}% mejor que usar promedios simples. {analogia}"
    elif veredicto == "AMARILLO":
        analogia = "Similar a un pronóstico de lluvia - útil para planear, pero lleva sombrilla por si acaso."
        return base + f"el modelo tiene capacidad predictiva limitada ({mejora_pct:.0f}% mejor que promedios), adecuado para análisis exploratorios pero no para decisiones críticas. {analogia}"
    else:
        analogia = "Como un mapa desactualizado - mejor buscar herramientas más precisas."
        return base + f"el modelo no supera significativamente las predicciones básicas ({mejora_pct:.0f}% mejora), recomendando revisar el enfoque. {analogia}"

print("\n" + "🔍 RESUMEN INTERPRETATIVO:")
print("="*40)
print(explicacion_breve(r2_test, nrmse_test, mejora_pct, veredicto))

In [ ]:
#############################
# Comparación con modelos regularizados
#############################
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

alphas = np.logspace(-3, 3, 25)

modelos = {
    "OLS": LinearRegression(),
    "Ridge": RidgeCV(alphas=alphas),
    "Lasso": LassoCV(alphas=alphas, cv=5, max_iter=5000, random_state=42),
    "ElasticNet": ElasticNetCV(l1_ratio=[0.2,0.5,0.8], alphas=alphas, cv=5, max_iter=5000, random_state=42)
}

res = []
series_por_modelo = {}  

base_feats = arreglar_despeje.get_feature_names_out(X_train.columns)

for nombre, base in modelos.items():
    pipe = Pipeline([("dropper", arreglar_despeje), ("model", base)])
    pipe.fit(X_train, y_train)

    # Métricas en TEST
    yhat = pipe.predict(X_test)
    r2 = r2_score(y_test, yhat)
    rmse = math.sqrt(mean_squared_error(y_test, yhat))
    mae = mean_absolute_error(y_test, yhat)
    res.append((nombre, r2, rmse, mae))

    # Coeficientes en una Serie; añadimos intercepto como fila aparte
    model_step = pipe.named_steps["model"]
    coef  = getattr(model_step, "coef_", None)
    inter = getattr(model_step, "intercept_", None)

    s = pd.Series(index=list(base_feats) + ["(intercepto)"], dtype=float)
    if coef is not None:
        s.loc[base_feats] = coef
    s.loc["(intercepto)"] = inter if inter is not None else np.nan
    series_por_modelo[nombre] = s

# Guardar métricas 
df_res = pd.DataFrame(res, columns=["modelo","R2","RMSE","MAE"]).sort_values("R2", ascending=False)
df_res.to_csv("comparacion_regularizacion.csv", index=False)
print(df_res)

df_coef_ancho = pd.DataFrame(series_por_modelo)  
df_coef_ancho = df_coef_ancho.reindex(columns=list(modelos.keys()))
orden_filas = ["(intercepto)"] + list(base_feats)
df_coef_ancho = df_coef_ancho.reindex(index=orden_filas)
df_coef_ancho.to_csv("coeficientes_modelos.csv", index_label="feature")

In [ ]:
############################################################
# Selección y guardado del mejor modelo
############################################################

import joblib, json, time, os, zipfile, io

best_row = df_res.loc[df_res["R2"].idxmax()]
best_name = best_row["modelo"]
print(f"🏆 MODELO GANADOR: {best_name}\n", best_row)

best_estimator = modelos[best_name]
best_pipeline = Pipeline([("dropper", arreglar_despeje), ("model", best_estimator)])
best_pipeline.fit(X_train, y_train)

buf = io.BytesIO()
joblib.dump(best_pipeline, buf)
buf.seek(0)

metadata = {
    "modelo_ganador": best_name,
    "R2_test": float(best_row["R2"]),
    "RMSE_test": float(best_row["RMSE"]),
    "MAE_test": float(best_row["MAE"]),
    "columnas_esperadas": X_train.columns.tolist(),
    "fecha_guardado": time.strftime("%Y-%m-%d %H:%M:%S")
}
meta_str = json.dumps(metadata, indent=2, ensure_ascii=False)

# --- Crear ZIP con todo adentro ---
dst_dir = "reg_lin_ganador"
os.makedirs(dst_dir, exist_ok=True)
zip_path = os.path.join(dst_dir, "reg_lin_ganador_bundle.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr(f"modelo_{best_name.lower()}.pkl", buf.getvalue())
    zf.writestr("metadata_modelo.json", meta_str)

print(f"✅ Bundle creado en: {zip_path}")
print("Contiene: [modelo_*.pkl, metadata_modelo.json]")
